In [1]:
import pandas as pd
import requests
import json
from sklearn.linear_model import LinearRegression

/home/sbhayani/miniconda3/envs/HSC/lib/python3.9/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


https://pantherdb.org/services/openAPISpec.jsp panther api doc

In [2]:
df15_r = (
    pd.read_csv('correlations/GEP_15_analyzed.csv')
    .drop('Unnamed: 0', axis=1).set_index('gene')
    .rename(columns={'GEP 15 (DNA Replication)':'GEP 15'})
)
df22_r = (
    pd.read_csv('correlations/GEP_22_analyzed.csv')
    .drop('Unnamed: 0', axis=1).set_index('gene')
    .rename(columns={'GEP 22 (Cell cycle)': 'GEP 22'})
)

In [3]:
hsc_15_lr = LinearRegression().fit(df15_r[['r hsc']], df15_r['GEP 15'])
prog_15_lr = LinearRegression().fit(df15_r[['r prog']], df15_r['GEP 15'])
hsc_22_lr = LinearRegression().fit(df22_r[['r hsc']], df22_r['GEP 22'])
prog_22_lr = LinearRegression().fit(df22_r[['r prog']], df22_r['GEP 22'])

In [4]:
def pred_GEP_expression(df, model, cell_type, GEP):
    assert cell_type in ['hsc', 'prog']
    assert GEP in [15, 22]
    
    return df.assign(**{f'GEP {GEP} pred based on r {cell_type}': model.predict(df[[f'r {cell_type}']])})

In [5]:
df15 = (df15_r
        .pipe(pred_GEP_expression, hsc_15_lr, 'hsc', 15)
        .pipe(pred_GEP_expression, prog_15_lr, 'prog', 15)
       )
df22 = (df22_r
        .pipe(pred_GEP_expression, hsc_22_lr, 'hsc', 22)
        .pipe(pred_GEP_expression, prog_22_lr, 'prog', 22)
       )

In [6]:
def get_panther_lists(df, GEP):
    # returns tuple of DFs to put into panther
    # filter only for regression significantly overshooting/undershooting
    # (HSC overshoot, HSC undershoot, prog overshoot, prog undershoot)
    
    assert GEP in [15, 22]
    p_cutoff = 0.05
    
    df_hsc = df[df[f'adj regression p of GEP {GEP} vs r hsc'] < p_cutoff]
    df_hsc_over = df_hsc[df_hsc[f'GEP {GEP} pred based on r hsc'] > df_hsc[f'GEP {GEP}']]
    df_hsc_under = df_hsc[df_hsc[f'GEP {GEP} pred based on r hsc'] < df_hsc[f'GEP {GEP}']]
    
    df_prog = df[df[f'adj regression p of GEP {GEP} vs r prog'] < p_cutoff]
    df_prog_over = df_hsc[df_hsc[f'GEP {GEP} pred based on r prog'] > df_hsc[f'GEP {GEP}']]
    df_prog_under = df_hsc[df_hsc[f'GEP {GEP} pred based on r prog'] < df_hsc[f'GEP {GEP}']]
    
    return df_hsc_over, df_hsc_under, df_prog_over, df_prog_under

In [7]:
df15_hsc_over, df15_hsc_under, df15_prog_over, df15_prog_under = df15.pipe(get_panther_lists, 15)
df22_hsc_over, df22_hsc_under, df22_prog_over, df22_prog_under = df22.pipe(get_panther_lists, 22)

In [8]:
def get_panther_list(df):
    return ','.join(df.index.to_list())

In [15]:
lst = 'C1orf54, CCR9, CD79A, CD79B, CFP, CMTM2, CMTM8, CTSG, EGR4, FCMR, GIMAP1, HLA.DRB5, IRF7, JAML, LILRA1, LILRB2, LYZ, MECOM, MZB1, NOL6, P2RY13, PPFIBP1, PTPRE, RNU6.1208P, SETBP1, TIAM2, TMEM229B, TMEM92, TNFAIP2, TNFRSF1B, ZNF46'

In [16]:
organism = "9606"

geneinfo_url = "https://pantherdb.org/services/oai/pantherdb/geneinfo"

payload = {
    "geneInputList": lst,
    "organism": organism
}

response = requests.post(geneinfo_url, data=payload)
if response.status_code != 200:
    print(f"Error fetching gene info: {response.status_code} {response.text}")
    exit()

data = response.json()

Querying gene info from PANTHER...


In [22]:
data

{'search': {'product': {'source': 'PANTHERDB', 'version': 19},
  'unmapped_list': {'unmapped': ['HLA.DRB5', 'RNU6.1208P']},
  'mapped_genes': {'gene': [{'family_id': 'PTHR19134',
     'sf_id': 'PTHR19134:SF499',
     'persistent_id': '',
     'annotation_type_list': {'annotation_data_type': [{'annotation_list': {'annotation': {'name': 'protein tyrosine phosphatase activity',
          'id': 'GO:0004725'}},
        'release_version': 19,
        'content': 'ANNOT_TYPE_ID_PANTHER_GO_SLIM_MF'},
       {'annotation_list': {'annotation': {'name': 'protein phosphatase',
          'id': 'PC00195'}},
        'release_version': 19,
        'content': 'ANNOT_TYPE_ID_PANTHER_PC'},
       {'annotation_list': {'annotation': [{'name': 'phosphoprotein phosphatase activity',
           'id': 'GO:0004721'},
          {'name': 'protein tyrosine phosphatase activity',
           'id': 'GO:0004725'},
          {'name': 'transmembrane receptor protein tyrosine phosphatase activity',
           'id': 'GO:00